# autoprof

> Functionality for running autoprof to measure the ICL in images of galaxy clusters.

In [7]:
# | default_exp autoprof

In [12]:
# | exporti

import logging
import os
import re
from pathlib import Path
import subprocess
from textwrap import dedent

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy.io import fits
from astropy.nddata import CCDData
from astropy.stats import sigma_clip
from astropy.visualization import simple_norm
from matplotlib.patches import Circle, Ellipse
from photutils.aperture import CircularAnnulus, EllipticalAnnulus
# from scipy.stats import median_abs_deviation

from nicl.background import get_background

In [13]:
# | export


def create_bkgsub_images(
    image_paths,
    background_mask_path,
    output_background_dir,
    temp_cleaned_dir,
    box_size,
    filter_size,
    label=None,
    savebkg=True,
    clean_nans=False,
):
    """Prepare background-subtracted images for autoprof.

    Loads images, applies the measurement mask to estimate background,
    saves the background maps, and writes cleaned background-subtracted images with NaNs set to 99.
    """
    Path(output_background_dir).mkdir(parents=True, exist_ok=True)
    Path(temp_cleaned_dir).mkdir(parents=True, exist_ok=True)

    background_mask = fits.getdata(background_mask_path).astype(bool)

    filenames_cleaned = {}

    for band, image_path in image_paths.items():
        image = CCDData.read(image_path, unit="adu")

        bkg = get_background(
            image.data,
            mask=background_mask,
            box_size=box_size,
            filter_size=filter_size,
        )

        # Save background
        bkg_filename = Path(output_background_dir) / f"{label}_bkg.fits"

        if savebkg:
            fits.writeto(
                bkg_filename,
                bkg.background,
                header=image.wcs.to_header(),
                overwrite=True,
            )

        # Subtract background and clean NaNs
        image_bkg_sub = image.data - bkg.background

        if clean_nans:
            image_bkg_sub = np.where(np.isfinite(image_bkg_sub), image_bkg_sub, 99)
            cleaned_filename = Path(temp_cleaned_dir) / f"{label}_cleaned_bkgsub.fits"

        else:
            cleaned_filename = Path(temp_cleaned_dir) / f"{label}_bkgsub.fits"

        cleaned_ccd = CCDData(image_bkg_sub, unit="adu", wcs=image.wcs)
        cleaned_ccd.write(cleaned_filename, overwrite=True)

        filenames_cleaned[band] = str(cleaned_filename)

    return filenames_cleaned

In [14]:
# | export


def run_autoprof(
    ids,
    image_files,
    mask_files=None,
    mode="image list",
    unit_type="intensity",
    gscale=0.4,
    pixelscale=0.3,
    zeropoint=23.9,
    out_dir="./",
    config_name="basic_config.py",
    fourier_orders=None,
    forced_photometry=False,
    forced_profile_filter=None,
    forced_profile_path=None,  # .prof file path forced forced photometry "dir/x.prof"
):
    """Run AutoProf with optional fixed photometry.

    If forced_photometry=True, the reference profile in forced_profile_path is used,
    and the pipeline steps are adjusted accordingly.
    """
    logger = logging.getLogger(__name__)
    os.makedirs(out_dir, exist_ok=True)
    config_file = os.path.join(out_dir, config_name)

    # Modify pipeline based on fixed photometry
    if forced_photometry:
        if not forced_profile_path:
            raise ValueError(
                "forced_profile_path must be provided when forced_photometry=True."
            )
        if not forced_profile_filter:
            raise ValueError(
                "forced_profile_filter must be provided when forced_photometry=True."
            )

        pipeline_steps = [
            "background",
            "psf",
            "center forced",
            "isophoteinit forced",
            "isophotefit forced",
            "isophoteextract forced",
            "writeprof",
        ]

    else:
        pipeline_steps = [
            "background",
            "psf",
            "center",
            "isophoteinit",
            "isophotefit",
            "isophoteextract",
            "writeprof",
        ]
    # FIXME: Should background be removed from pipeline steps?

    if mask_files is not None:
        pipeline_steps.insert(0, "mask segmentation map")

    # Write AutoProf config
    with open(config_file, "w") as f:
        f.write(
            dedent(f"""\
import numpy as np
ap_process_mode = "{mode}"
ap_name = {ids}
ap_image_file = {image_files}
""")
        )
        if mask_files is not None:
            f.write(f"ap_mask_file = {mask_files}\n")

        f.write(
            dedent(f"""\
ap_saveto = "{out_dir}"
ap_pixscale = {pixelscale}
ap_zeropoint = {zeropoint}
ap_samplegeometricscale = {gscale}
ap_doplot = True
ap_extractfull = True
ap_fluxunits = "{unit_type}"
ap_isoclip = True
ap_isoclip_nsigma = 5
ap_ellipsefit = True
ap_fix_pa = False
ap_initial_pa = 45.0 
ap_fix_ellipticity = False
ap_initial_ellipticity = 0.3
""")
        )

        if forced_photometry:
            f.write(f'ap_forcing_profile = "{forced_profile_path}"\n')

        if fourier_orders:
            f.write(f"ap_iso_measurecoefs = {fourier_orders}\n")

        f.write(f"ap_new_pipeline_steps = {pipeline_steps}\n")

    # Run AutoProf
    log_file = Path(out_dir) / "AutoProf.log"
    env = os.environ.copy()
    try:
        del env["MPLBACKEND"]
    except KeyError:
        pass
    with open(log_file, "w") as log:
        process = subprocess.run(
            ["autoprof", config_name],
            cwd=out_dir,
            stderr=subprocess.STDOUT,
            stdout=log,
            text=True,
            # shell=True,
            env=env,
        )

    if process.returncode == 0:
        logger.info(f"AutoProf completed successfully for {ids}!")
    else:
        logger.error(f"AutoProf failed for {ids}.")
        raise RuntimeError(f"AutoProf failed, see {log_file} for details.")

In [ ]:
# | export


def Extract_SB_using_AP_shapes(
    image_path,
    object_mask_path=None,
    profile_path=None,
    bcg_pos=None,
    annuli_shape="elliptical",
    pixelscale=0.3,
    core_mask_path=None,
    rad_limit_annulus=None,
    verbose=None,
    add_in_AP_background_level=False,
    num_points=1,
    output_csv_path=None,
    show_plot=True,
    plot_output_path=None,
    prefix=None,
    plot_lims=1000,
):
    """Extract surface-brightness profiles from an image using the output from AutoProf.

    Compute flux statistics for elliptical or circular annuli centred either at bcg_pos (SkyCoord)
    or at image centre if bcg_pos is not given.
    """
    logger = logging.getLogger(__name__)
    profile = pd.read_csv(profile_path, skiprows=1)

    ccd = CCDData.read(image_path, unit="adu")
    image = ccd.data
    wcs = ccd.wcs

    image_height, image_width = image.shape

    # If bcg position is not provided image centre is taken as the centre.
    if bcg_pos is not None:
        x, y = wcs.world_to_pixel(bcg_pos)
    else:
        x, y = image_width / 2, image_height / 2

    # Loading mask if provided, otherwise use a blank mask
    if object_mask_path is not None:
        object_mask = fits.getdata(object_mask_path).astype(bool)
    else:
        object_mask = np.zeros_like(image, dtype=bool)

    if core_mask_path is not None:
        core_mask = fits.getdata(core_mask_path).astype(bool)
        combined_mask = object_mask | core_mask | ~np.isfinite(image)
    else:
        combined_mask = object_mask | ~np.isfinite(image)

    masked_image = np.where(combined_mask, np.nan, image)

    profile["R"] = profile["R"] / pixelscale  # Convert arcsec to pixels

    if rad_limit_annulus:
        profile = profile[profile["R"] < rad_limit_annulus]

    rad = profile["R"].values
    ellip = profile["ellip"].values
    pa = profile["pa"].values

    selected_annuli = sorted(set(rad))
    flux_stats = []
    problematic_annuli = []

    if add_in_AP_background_level:
        with open(str(profile_path).replace(".prof", ".aux")) as f:
            content = f.read()
        # Extracting background value and error from AP aux file -- which is removed in isophotal flux profiles in AP.
        background_match = re.search(
            r"background:\s*([-\d.e+]+)\s*\+\-\s*([\d.e+-]+)", content
        )
        if background_match:
            background_value = float(background_match.group(1))
            # background_error = float(background_match.group(2))
        else:
            background_value = 0
            # background_error = 0

        # Get pixel scale from .aux or use provided one
        pixscale_match = re.search(r"option ap_pixscale:\s*([\d.e+-]+)", content)
        if pixscale_match:
            ap_pixscale = float(pixscale_match.group(1))
        else:
            ap_pixscale = pixelscale

        AP_background_level = background_value / ap_pixscale**2
        # AP_background_level_err = background_error / ap_pixscale**2

    else:
        AP_background_level = 0

    logger.debug(
        f"The background level to add in isophote measurements will be: {AP_background_level}"
    )

    for i in range(len(selected_annuli) - 1):
        r_inner = selected_annuli[i]
        r_outer = selected_annuli[i + 1]
        e = ellip[i]
        theta = np.deg2rad(pa[i])

        if annuli_shape == "circular":
            annulus = CircularAnnulus((x, y), r_in=r_inner, r_out=r_outer)

        else:
            annulus = EllipticalAnnulus(
                (x, y),
                a_in=r_inner,
                a_out=r_outer,
                b_in=r_inner * (1 - e),
                b_out=r_outer * (1 - e),
                theta=theta
                - np.pi
                / 2,  # Convert from PA to photutils angle (the definition of theta is different in AP and photutils/matplotlib)
            )
        annulus_mask = annulus.to_mask(method="center")
        mask_image = annulus_mask.to_image(masked_image.shape)

        valid_flux_values = masked_image[np.isfinite(masked_image) & (mask_image > 0)]

        total_pixels = np.sum(mask_image > 0)
        total_valid = len(valid_flux_values)
        total_masked = total_pixels - total_valid

        if total_valid == 0:
            problematic_annuli.append(
                {"x": x, "y": y, "r_inner": r_inner, "r_outer": r_outer}
            )
            mean_flux = median_flux = std_flux = np.nan
            clipped_mean_flux = clipped_median_flux = clipped_std_flux = np.nan
        else:
            mean_flux = np.nanmean(valid_flux_values) / (pixelscale**2)
            median_flux = np.nanmedian(valid_flux_values) / (pixelscale**2)
            std_flux = (
                np.nanstd(valid_flux_values) if len(valid_flux_values) > 1 else np.nan
            )

            clipped = sigma_clip(
                valid_flux_values, sigma=3, cenfunc="median", maxiters=5
            )
            clipped_valid = clipped.data[~clipped.mask]

            clipped_mean_flux = np.nanmean(clipped_valid) / (pixelscale**2)
            clipped_median_flux = np.nanmedian(clipped_valid) / (pixelscale**2)

            clipped_std_flux = (
                np.nanstd(clipped_valid) if len(clipped_valid) > 1 else np.nan
            )

        flux_stats.append(
            {
                "Centre_pixel": (x, y),
                "Inner_radius_pix": r_inner,
                "Outer_radius_pix": r_outer,
                "Inner_radius_arcsec": r_inner * pixelscale,
                "Outer_radius_arcsec": r_outer * pixelscale,
                "SMA_annulus_centre_pix": (r_inner + r_outer) / 2,
                "SMA_annulus_centre_arcsec": (
                    (r_inner * pixelscale) + (r_outer * pixelscale)
                )
                / 2,
                "Mean_flux_annulus": mean_flux,
                "Median_flux_annulus": median_flux,
                "Std_flux_annulus": std_flux,
                "Total_valid_pix_annulus": total_valid,
                "Total_masked_pix_annulus": total_masked,
                "Clipped_mean_flux_annulus": clipped_mean_flux,
                "Clipped_median_flux_annulus": clipped_median_flux,
                "Std_clipped_flux_annulus": clipped_std_flux,
            }
        )

    if show_plot:
        fig, ax = plt.subplots(figsize=(6, 6))

        ax.imshow(
            image,
            origin="lower",
            cmap="gray",
            norm=simple_norm(masked_image, "sqrt", percent=80),
        )

        # Mask overlay
        mask_overlay = np.ma.masked_where(~combined_mask, combined_mask.astype(float))
        ax.imshow(mask_overlay, origin="lower", cmap="Reds", vmin=0, vmax=1, alpha=0.8)

        for i in range(len(selected_annuli) - 1):
            r_outer = selected_annuli[i + 1]
            e = ellip[i]
            theta = np.deg2rad(pa[i])

            if annuli_shape == "circular":
                circle = Circle(
                    (x, y), radius=r_outer, edgecolor="blue", facecolor="none", lw=1
                )
                ax.add_patch(circle)
            else:
                ellipse = Ellipse(
                    (x, y),
                    width=2 * r_outer,
                    height=2 * r_outer * (1 - e),
                    angle=np.rad2deg(theta - np.pi / 2),
                    edgecolor="red",
                    facecolor="none",
                    lw=0.5,
                )
                ax.add_patch(ellipse)

        ax.set_title("Annuli Overlay")
        plt.show()

    flux_stats_df = pd.DataFrame(flux_stats)
    return flux_stats_df, profile, problematic_annuli, AP_background_level